# Lezione 16 — Embedding: vettori densi al posto dei conteggi

**Come si usa questo notebook.** Come sempre. Prerequisito: Lezione 15
(tokenizzazione, vocabolario, bag of words). Lì il classificatore ha
saltato al ~95% leggendo *quali* parole compaiono in una memoria, ma il
bag of words tratta ogni parola come un compartimento stagno: "treno" e
"ferrovia" sono due colonne indipendenti, senza nessuna nozione che si
somigliano.

**Cosa saprai fare alla fine:** sostituire il bag of words con vettori
densi imparati dal modello stesso (l'`Embedding` layer di Keras),
trasformare una sequenza di vettori-parola in un unico vettore-frase con il
*mean pooling*, e confrontare onestamente il risultato con la baseline
della Lezione 15.

## Parte 1 — Teoria: dal conteggio al vettore denso

Il bag of words (Lezione 15) rappresenta una parola come **una colonna** in
un vettore lungo quanto il vocabolario: sparso (quasi tutti zeri), enorme
con vocabolari veri (centinaia di migliaia di parole), e senza nessuna
nozione di somiglianza — la distanza tra "treno" e "ferrovia" è identica
alla distanza tra "treno" e "banana".

Un **embedding** rappresenta invece ogni parola come un vettore **denso**
(tutti i valori diversi da zero) e **corto** (tipicamente 8-300 numeri,
molto meno del vocabolario). I valori non sono scelti a mano: sono **pesi
del modello**, inizializzati a caso e aggiornati dalla backpropagation
esattamente come i pesi di un `Dense` (Lezione 10-11). Il modello impara
durante il training che le parole usate in contesti simili devono avere
vettori vicini — non perché qualcuno gli dice il significato delle parole,
ma perché minimizzare la loss lo spinge a farlo (se "treno" e "ferrovia"
compaiono spesso in frasi che portano alla stessa etichetta, il gradiente
li tira verso la stessa zona dello spazio).

In Keras, l'`Embedding` layer è essenzialmente una **tabella di lookup**:
una matrice di forma `(vocabolario, dimensione_embedding)`, dove la riga
`i` è il vettore appreso per il token con id `i`. Il layer prende in
ingresso una sequenza di id di token (interi, come quelli prodotti da
`TextVectorization(output_mode='int')` nella Lezione 15) e restituisce la
sequenza dei vettori corrispondenti — una riga della tabella per ogni
token.

Due parametri principali:

- `input_dim` — quante righe ha la tabella, cioè la dimensione del
  vocabolario (incluso il token di padding e l'OOV);
- `output_dim` — quante colonne, cioè la dimensione di ogni vettore-parola
  (un iperparametro: più alto cattura più sfumature ma costa più parametri
  e più dati per essere stimato bene).

Un dettaglio che conta piu' di quanto sembri: `output_sequence_length`
(Lezione 15) porta tutte le sequenze alla stessa lunghezza aggiungendo
padding (id `0`) alle frasi piu' corte — e con frasi brevi in sequenze
lunghe, il padding puo' essere la **maggioranza** dei token. Se
l'aggregazione a valle (tra poco) media anche i vettori del padding, il
risultato si diluisce con lo stesso vettore ripetuto molte volte, per
ogni frase. `Embedding(..., mask_zero=True)` risolve il problema:
propaga una **maschera** (quali posizioni sono padding) ai layer
successivi, che la rispettano ignorando quelle posizioni invece di
includerle nel calcolo — lo userai da qui in poi ogni volta che aggreghi
una sequenza mascherata.

Una frase di `L` parole diventa quindi una matrice `L x output_dim`, non un
singolo vettore: il layer successivo deve **aggregare** questa sequenza in
qualcosa che un `Dense` possa consumare (che si aspetta un vettore fisso,
non una sequenza di lunghezza variabile). La strategia più semplice è il
**mean pooling**: la media dei vettori-parola della frase, riga per riga.
`GlobalAveragePooling1D` fa esattamente questo. È un'aggregazione grezza
(butta via l'ordine, come il bag of words), ma è un primo passo onesto:
la prossima lezione (sentence embeddings) approfondisce le alternative.

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import keras
from keras.layers import TextVectorization

keras.utils.set_random_seed(42)
print('Keras', keras.__version__)

Keras 3.15.0


In [2]:
frasi_demo = np.array(['Marco visited Glasgow with his son.',
                       'The user prefers short updates.'])

vettorizzatore_demo = TextVectorization(max_tokens=20, output_mode='int',
                                        output_sequence_length=8)
vettorizzatore_demo.adapt(frasi_demo)
sequenze_demo = vettorizzatore_demo(frasi_demo)
print('sequenze di id (padded a 8):')
print(sequenze_demo.numpy())

embedding_demo = keras.layers.Embedding(input_dim=20, output_dim=4)
vettori_demo = embedding_demo(sequenze_demo)
print('\nforma dopo Embedding:', vettori_demo.shape, '(frasi, token, dimensione_embedding)')
print('vettore denso del primo token della prima frase:')
print(vettori_demo[0, 0].numpy())

sequenze di id (padded a 8):
[[10  3 12  2 11  7  0  0]
 [ 6  4  9  8  5  0  0  0]]



forma dopo Embedding: (2, 8, 4) (frasi, token, dimensione_embedding)
vettore denso del primo token della prima frase:
[-0.04573692  0.00636358 -0.01658783 -0.01131532]


2026-07-16 21:15:03.549595: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Leggi l'output: `TextVectorization` con `output_sequence_length=8` produce
sequenze di lunghezza fissa (padding con zeri se la frase è più corta) —
necessario perché `Embedding` (e i layer dopo) lavorano su tensori di forma
fissa, non su liste di lunghezza variabile. `Embedding` trasforma ogni id
intero in un vettore di 4 numeri: la tabella di lookup ha forma `(20, 4)`
(20 righe = `input_dim`, una per ogni id possibile nel vocabolario; 4
colonne = `output_dim`). Nota che il vettore del token di padding (id 0) è
comunque un vettore appreso come gli altri — non è automaticamente zero.
Qui `embedding_demo` non usa `mask_zero=True` apposta, per tenere
l'esercizio sulla sola forma; il passo del progetto (Parte 3) lo riattiva,
e li' la differenza si vede eccome.

## Parte 2 — Esercizio guidato

Il tuo compito: applica `GlobalAveragePooling1D` a `vettori_demo` e
verifica che la forma finale sia `(2, 4)` — un vettore denso per ogni
frase, non più per ogni token.

In [3]:
# Scrivi qui: applica keras.layers.GlobalAveragePooling1D() a vettori_demo
# e stampa la forma del risultato.

pass

### Soluzione spiegata

`GlobalAveragePooling1D()` fa la media lungo l'asse dei token (asse 1):
da `(2, 8, 4)` (frasi, token, dimensione) a `(2, 4)` (frasi, dimensione).
Ogni frase è ora un singolo vettore denso, pronto per un `Dense` — lo
stesso ruolo che il bag of words multi-hot aveva nella Lezione 15, ma
denso invece che sparso, e con la somiglianza tra parole codificata nei
valori invece che ignorata.

In [4]:
pooling_demo = keras.layers.GlobalAveragePooling1D()
frasi_pooled_demo = pooling_demo(vettori_demo)
print('forma dopo il pooling:', frasi_pooled_demo.shape)
assert frasi_pooled_demo.shape == (2, 4)

forma dopo il pooling: (2, 4)


## Parte 3 — Il progetto: Memory AI Lab, passo 16 — embedding al posto del bag of words

La Lezione 15 ha portato il classificatore dal ~60-65% (feature povere)
al ~95% (bag of words) leggendo le parole delle memorie. Oggi sostituiamo
il bag of words con `Embedding` + `GlobalAveragePooling1D`: stesso
protocollo (vocabolario costruito sul solo train), stessa rete a valle
(`Dense(32, relu)` + `Dropout(0.3)` + `Dense(4, softmax)`), la sola
rappresentazione del testo cambia.

Non aspettarti un salto enorme di accuratezza — il bag of words era già a
un ottimo punto per un dataset piccolo come questo. Il punto della lezione
non è "più accurato", è **cosa rappresenta il vettore**: con l'embedding,
parole usate in contesti simili nel train finiscono vicine nello spazio
appreso, una proprietà che il bag of words non ha e che tornerà utile per
la ricerca semantica (Lezioni 17-18).

In [5]:
from pathlib import Path

processed = Path('..') / 'datasets' / 'processed'
memorie = {n: pd.read_csv(processed / f'memory_{n}.csv') for n in ['train', 'val']}
classi = ['episodic', 'preference', 'semantic', 'unknown']
mappa = {c: i for i, c in enumerate(classi)}
testi = {n: f['text'].astype(str).to_numpy() for n, f in memorie.items()}
target = {n: f['type'].map(mappa).to_numpy() for n, f in memorie.items()}

LUNGHEZZA_SEQUENZA = 24  # copre la quasi totalita' delle memorie del dataset
vettorizzatore = TextVectorization(max_tokens=300, output_mode='int',
                                   output_sequence_length=LUNGHEZZA_SEQUENZA)
vettorizzatore.adapt(testi['train'])                 # vocabolario dal SOLO train
X_seq = {n: vettorizzatore(t).numpy() for n, t in testi.items()}
print('sequenze:', X_seq['train'].shape, '(memorie x id di token, padded)')

sequenze: (213, 24) (memorie x id di token, padded)


In [6]:
keras.utils.set_random_seed(42)
lettore_embedding = keras.Sequential([
    keras.layers.Embedding(input_dim=300, output_dim=16, mask_zero=True),
    keras.layers.GlobalAveragePooling1D(),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(4, activation='softmax'),
])
lettore_embedding.compile(loss='sparse_categorical_crossentropy', optimizer='adam',
                          metrics=['accuracy'])
lettore_embedding.fit(X_seq['train'], target['train'], epochs=300,
                      validation_data=(X_seq['val'], target['val']),
                      callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=20,
                                                               restore_best_weights=True)],
                      verbose=0)

acc_embedding = lettore_embedding.evaluate(X_seq['val'], target['val'], verbose=0)[1]
print('bag of words (Lezione 15) : ~95% su validation')
print(f'embedding + mean pooling  : {acc_embedding:.0%} su validation')

bag of words (Lezione 15) : ~95% su validation
embedding + mean pooling  : 95% su validation


In [7]:
# Vicinanza semantica: la tabella di embedding ha imparato qualcosa sulle parole?
matrice_embedding = lettore_embedding.layers[0].get_weights()[0]
vocabolario = vettorizzatore.get_vocabulary()

def parole_vicine(parola, k=5):
    if parola not in vocabolario:
        return f'"{parola}" non e\' nel vocabolario del train'
    idx = vocabolario.index(parola)
    vettore = matrice_embedding[idx]
    distanze = np.linalg.norm(matrice_embedding - vettore, axis=1)
    vicini = np.argsort(distanze)[1:k + 1]  # [0] e' la parola stessa
    return [(vocabolario[i], round(float(distanze[i]), 3)) for i in vicini]

for parola in ['visited', 'prefers', 'likes']:
    print(parola, '->', parole_vicine(parola))

visited -> [(np.str_('weekend'), 0.16), (np.str_('long'), 0.186), (np.str_('call'), 0.195), (np.str_('had'), 0.201), (np.str_('finished'), 0.229)]
prefers -> [(np.str_('short'), 0.365), (np.str_('updates'), 0.39), (np.str_('user'), 0.392), (np.str_('sessions'), 0.471), (np.str_('morning'), 0.542)]
likes -> [(np.str_('meetings'), 0.146), (np.str_('walking'), 0.155), (np.str_('dislikes'), 0.227), (np.str_('morning'), 0.237), (np.str_('late'), 0.245)]


Guarda l'output con occhio critico, non entusiasta: con un dataset di
poche centinaia di memorie il modello vede ogni parola pochissime volte, e
gli embedding imparati da zero (senza pre-training) hanno bisogno di
**molti** dati per organizzarsi in modo pulito — potresti trovare vicinanze
sensate per le parole più frequenti e rumore per le altre. Non è un bug:
è il limite onesto dell'embedding "from scratch" su un dataset piccolo, e
il motivo per cui in pratica si usano spesso embedding **pre-addestrati**
su corpora enormi (fuori scope qui: niente credenziali Kaggle/HuggingFace
in questo ambiente, ma il principio del *transfer learning* torna nel
modulo Fase 4 del corso). Con più dati o più epoche la struttura migliora,
ma il punto pedagogico resta: l'embedding è denso e imparabile, il bag of
words no.

## Cosa hai imparato

- Un **embedding** è una tabella di lookup appresa: `input_dim` righe
  (vocabolario), `output_dim` colonne (dimensione del vettore), pesi
  aggiornati dalla backpropagation come un `Dense` qualunque.
- Una frase diventa una **sequenza** di vettori-parola: serve
  un'aggregazione (`GlobalAveragePooling1D`, media lungo i token) per
  ottenere un vettore fisso da dare a un `Dense`.
- L'embedding è denso e può catturare somiglianza tra parole; il bag of
  words è sparso e tratta ogni parola come indipendente dalle altre.
- Con pochi dati, un embedding addestrato da zero organizza lo spazio in
  modo imperfetto — non è una contraddizione della teoria, è un limite dei
  dati disponibili.

## Quiz

1. Perché una frase, dopo `Embedding`, è una sequenza di vettori e non un
   singolo vettore? Cosa risolve `GlobalAveragePooling1D`?
2. Qual e' la differenza tra `input_dim` e `output_dim` in
   `keras.layers.Embedding`?
3. Un vocabolario di 50.000 parole con `output_dim=16` produce quanti
   parametri appresi solo nella tabella di embedding? Confronta con un
   bag of words di pari vocabolario dato in input a un `Dense(32)`.

<details>
<summary><b>Apri le risposte</b></summary>

1. `Embedding` sostituisce ogni id di token con il suo vettore, quindi una
   frase di `L` token diventa una matrice `L x output_dim`, non un
   vettore. `GlobalAveragePooling1D` aggrega quella sequenza facendo la
   media lungo l'asse dei token, producendo un vettore fisso `output_dim`
   che un `Dense` può consumare.
2. `input_dim` e' il numero di righe della tabella (quanti id di token
   distinti esistono, cioè la dimensione del vocabolario); `output_dim` e'
   il numero di colonne (quanti numeri ha ciascun vettore-parola appreso).
3. La tabella di embedding ha `50000 x 16 = 800.000` parametri. Un bag of
   words di 50.000 colonne dato in input a un `Dense(32)` avrebbe
   `50000 x 32 + 32 = 1.600.032` parametri nel solo primo layer — di più,
   e senza nessuna nozione di somiglianza tra parole incorporata nei pesi.
</details>

## Fonti

- Keras, `Embedding` layer:
  https://keras.io/api/layers/core_layers/embedding/
- Keras, `GlobalAveragePooling1D`:
  https://keras.io/api/layers/pooling_layers/global_average_pooling1d/
- TensorFlow, *Word embeddings*:
  https://www.tensorflow.org/text/guide/word_embeddings

## Prossima lezione

Il mean pooling è la strategia più semplice per passare da parole a
frase: la prossima lezione guarda più da vicino cosa significa avere un
**vettore per frase** (non solo per parola), come si confrontano due
vettori-frase, e dove si collocano gli encoder di frase pre-addestrati
che qui non usiamo.